# Lab 05-04 · Jupyter, Testing and Reproducible Workflows

Companion to chapter [05-04](../../chapters/05-04-reproducible-workflows.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running. Everything here runs in a temporary folder, so nothing in your repository is touched, and git uses a fixed author and clock so that your hashes match the handout.

In [1]:
import os
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

import nbformat

LAB_DIR = Path.cwd()
TOOLS = (LAB_DIR / "../../../../tools").resolve()       # this repository's tools folder
work = Path(tempfile.mkdtemp())
os.chdir(work)
print("working in a temporary folder:", work.name.startswith("tmp"))

working in a temporary folder: True


## Part 1 — Hidden state, caught

The notebook below was saved after its cells ran in the order 1, 2, 4, 3: the resistor was changed to 2 kΩ *before* τ was computed, but the change sits below the computation on the page.

**🔮 Predict:** the saved output of cell 3 says τ = 2.000 ms. What will a fresh, top-to-bottom run print?

In [2]:
saved = nbformat.v4.new_notebook()
saved.cells = [nbformat.v4.new_code_cell(src) for src in
               ("r = 1000", "c = 1e-6", "tau = r * c\nprint(f'tau = {tau * 1e3:.3f} ms')", "r = 2000")]
for cell, count in zip(saved.cells, [1, 2, 4, 3]):
    cell.execution_count = count
saved.cells[2].outputs = [nbformat.v4.new_output("stream", name="stdout", text="tau = 2.000 ms\n")]

nbformat.write(saved, "saved.ipynb")
# Re-execute in a brand-new kernel, top to bottom: the command-line equivalent is
#   jupyter nbconvert --to notebook --execute saved.ipynb --output fresh.ipynb
subprocess.run([sys.executable, "-m", "nbconvert", "--to", "notebook", "--execute", "saved.ipynb",
                "--output", "fresh.ipynb"], capture_output=True, check=True)
fresh = nbformat.read("fresh.ipynb", as_version=4)
for label, book in (("saved", saved), ("fresh", fresh)):
    print(f"{label}: {book.cells[2].outputs[0]['text'].strip():16} counts {[c.execution_count for c in book.cells]}")

saved: tau = 2.000 ms   counts [1, 2, 4, 3]
fresh: tau = 1.000 ms   counts [1, 2, 3, 4]


The counts 1, 2, 4, 3 were the clue; the fresh run, which knows nothing except the code on the page, prints τ = 1.000 ms with counts 1, 2, 3, 4. This is exactly what this repository's `tools/run_notebooks.py` does to every lab on every change.

## Part 2 — Tests that find bugs

A resistor's first two color bands are digits and the third is a power-of-ten multiplier. Below are a correct decoder and three broken ones, and four tests.

**🔮 Predict:** for each broken version, which of the four tests fail?

In [3]:
COMMON = '''
DIGITS = {"black": 0, "brown": 1, "red": 2, "orange": 3, "yellow": 4,
          "green": 5, "blue": 6, "violet": 7, "grey": 8, "white": 9}

def digit(color):
    if color not in DIGITS:
        raise ValueError(f"unknown color: {color}")
    return DIGITS[color]
'''
VERSIONS = {
    "correct":            "def decode(b1, b2, m):\n    return (10 * digit(b1) + digit(b2)) * 10 ** digit(m)\n",
    "multiplier off by 1": "def decode(b1, b2, m):\n    return (10 * digit(b1) + digit(b2)) * 10 ** (digit(m) + 1)\n",
    "digits swapped":     "def decode(b1, b2, m):\n    return (10 * digit(b2) + digit(b1)) * 10 ** digit(m)\n",
    "no validation":      "def decode(b1, b2, m):\n    d = lambda c: DIGITS.get(c, 0)\n    return (10 * d(b1) + d(b2)) * 10 ** d(m)\n",
}
Path("test_decoder.py").write_text(textwrap.dedent('''
    import pytest
    from decoder import decode

    def test_brown_black_red_is_1k():
        assert decode("brown", "black", "red") == 1_000

    def test_yellow_violet_orange_is_47k():
        assert decode("yellow", "violet", "orange") == 47_000

    def test_red_red_black_is_22():
        assert decode("red", "red", "black") == 22

    def test_unknown_color_rejected():
        with pytest.raises(ValueError):
            decode("pink", "black", "red")
'''))

def pytest_run(*files):
    env = {**os.environ, "COLUMNS": "80", "PYTHONDONTWRITEBYTECODE": "1"}
    result = subprocess.run([sys.executable, "-m", "pytest", "-qq", "--tb=no", "-rf", "--color=no", "-p", "no:cacheprovider", *files],
                            capture_output=True, text=True, env=env)
    return [line.split("::")[1].split(" ")[0] for line in result.stdout.splitlines() if line.startswith("FAILED")]

for name, body in VERSIONS.items():
    Path("decoder.py").write_text(COMMON + body)
    failed = pytest_run("test_decoder.py")
    print(f"{name:20} {len(failed)} failed: {', '.join(failed) or '-'}")

correct              0 failed: -


multiplier off by 1  3 failed: test_brown_black_red_is_1k, test_yellow_violet_orange_is_47k, test_red_red_black_is_22


digits swapped       2 failed: test_brown_black_red_is_1k, test_yellow_violet_orange_is_47k


no validation        1 failed: test_unknown_color_rejected


- **Multiplier off by one** fails all three known cases: every value is ten times too large.
- **Digits swapped** fails 1 kΩ (it reads 100 Ω) and 47 kΩ (74 kΩ), but *passes* 22 Ω: a repeated digit makes the swap invisible.
- **No validation** passes every numeric test and fails only the error test: "pink" silently becomes black, a plausible-looking wrong answer.

Each test catches a different class of mistake; none catches them all. That is why a suite combines several known cases (chosen so that they are *not* symmetric), plus error cases.

## Part 3 — Reading a git history

**🔮 Predict:** a wrong gain was committed somewhere in a three-commit history. Can git find the commit that introduced the text `20.0` without you reading every diff?

In [4]:
GIT_ENV = {**os.environ,
           "GIT_AUTHOR_NAME": "Ada Learner", "GIT_AUTHOR_EMAIL": "ada@example.com",
           "GIT_COMMITTER_NAME": "Ada Learner", "GIT_COMMITTER_EMAIL": "ada@example.com",
           "GIT_AUTHOR_DATE": "2026-11-18T09:00:00+00:00", "GIT_COMMITTER_DATE": "2026-11-18T09:00:00+00:00",
           "GIT_CONFIG_GLOBAL": os.devnull, "GIT_CONFIG_NOSYSTEM": "1"}

def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True, env=GIT_ENV)
    return (result.stdout + result.stderr).rstrip()

Path("amp").mkdir()
os.chdir("amp")
git("init", "-q", "-b", "main")
for message, files in (("Add amplifier model", {"gain.py": "gain = 2.0\n", "README.md": "Amplifier model\n"}),
                       ("Tidy README and update gain", {"gain.py": "gain = 20.0\n", "README.md": "Amplifier model (v2)\n"}),
                       ("Add bandwidth", {"bandwidth.py": "f_3db = 1e6\n"})):
    for name, text in files.items():
        Path(name).write_text(text)
    git("add", ".")
    git("commit", "-q", "-m", message)
print(git("log", "--oneline"))
print("introduced 20.0:", git("log", "-S", "20.0", "--format=%h %s"))
culprit = git("log", "-S", "20.0", "--format=%h")
print("gain.py before it:", git("show", f"{culprit}~1:gain.py"))
os.chdir("..")

7a68dd3 Add bandwidth
091cff7 Tidy README and update gain
f8209b3 Add amplifier model
introduced 20.0: 091cff7 Tidy README and update gain
gain.py before it: gain = 2.0


`git log -S "20.0"` (the "pickaxe") lists the commits that changed how many times the text appears: here, the second commit, whose message mentions the gain only in passing. `git show <commit>~1:gain.py` recovers the file as it was just before. Small commits with honest messages make this kind of search fast.

## Part 4 — Breaking a verify check

This repository's checker reads an answer key from Markdown and compares each published value with an independent computation, to half a unit in the last published digit.

**🔮 Predict:** the true parallel resistance of 10 kΩ and 4.7 kΩ is 3.19728 kΩ. If the key says 3.198 kΩ, will the check pass?

In [5]:
sys.path.insert(0, str(TOOLS))
from eeverify import Verifier

def check_key(published):
    Path("demo.md").write_text("<!-- answer-key -->\n| ID | Quantity | Value | Unit |\n|---|---|---|---|\n"
                               f"| D.1 | 10 kΩ ∥ 4.7 kΩ | {published} | kΩ |\n")
    v = Verifier.from_files("demo", "demo.md")
    v.check("D.1", 10 * 4.7 / (10 + 4.7))
    try:
        v.finish()
    except SystemExit as exit_status:
        print("checker exit status:", exit_status.code)

for published in ("3.197", "3.198", "3.20", "3.1973"):
    check_key(published)

  ok   D.1            10 kΩ ∥ 4.7 kΩ = 3.197 kΩ
PASS demo: 1 checks passed, 0 failed, 1 answer-key entries
  FAIL D.1            computed 3.197278912, published 3.198 (±0.0005)
FAIL demo: 0 checks passed, 1 failed, 1 answer-key entries
checker exit status: 1
  ok   D.1            10 kΩ ∥ 4.7 kΩ = 3.20 kΩ
PASS demo: 1 checks passed, 0 failed, 1 answer-key entries
  ok   D.1            10 kΩ ∥ 4.7 kΩ = 3.1973 kΩ
PASS demo: 1 checks passed, 0 failed, 1 answer-key entries


3.197 passes; 3.198 fails, because a correctly rounded value of 3.19728 is 3.197 and the allowed error is ±0.0005 kΩ = ±0.5 Ω. 3.20 passes (it is correctly rounded to two decimals) but checks less, and 3.1973 passes with a tighter ±0.05 Ω. The smallest error the checker can catch is set by how many digits are published.

## Your turn

1. In Part 2, write one more test that catches the swapped-digits bug *and* the multiplier bug with a single assertion. (Hint: choose bands whose digits differ and whose multiplier is not black.)
2. In Part 3, use `git diff <first-hash> HEAD -- gain.py` to see the total change to `gain.py` across the history.
3. In Part 4, find the published value with the fewest digits that the checker would still accept for 3.19728 kΩ, and explain why publishing it would be a poor choice.

In [6]:
os.chdir(LAB_DIR)                     # leave the temporary folder